In [1]:
!pip install torch


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [17]:
import torch

### **Standard normal CDF and inverse CDF**


$$
Z \sim N(0,1).
$$

Standard normal CDF 
$$
\Phi(z).
$$

Inverse CDF



$$
\Phi^{-1}(u).
$$

In [3]:
def normal_cdf(z):
    normal = torch.distributions.Normal(0.0, 1.0)
    return normal.cdf(z)


def normal_ppf(u):
    normal = torch.distributions.Normal(0.0, 1.0)
    return normal.icdf(u)

### **Gaussian copula density**

$$
c_\rho(u,v)
=\frac{1}{\sqrt{1-\rho^2}}\exp\left[-
\frac{\rho^2(z_u^2+z_v^2)-2\rho z_u z_v}{2(1-\rho^2)}\right]
$$

where


$$
z_u = \Phi^{-1}(u)
\qquad
z_v = \Phi^{-1}(v)
$$



In [4]:
def gaussian_copula_density_torch(u, v, rho):
    eps = 1e-6

    u = torch.clamp(u, eps, 1 - eps)
    v = torch.clamp(v, eps, 1 - eps)

    z_u = normal_ppf(u)
    z_v = normal_ppf(v)

    numerator = torch.exp(- (rho**2 * (z_u**2 + z_v**2) - 2 * rho * z_u * z_v) / (2 * (1 - rho**2)))
    denominator = torch.sqrt(1 - rho**2)

    return numerator / denominator

### **Gaussian conditional CDF**

$$
z_u = \Phi^{-1}(u),
\qquad
z_v = \Phi^{-1}(v),
$$


$$
H_\rho(u,v)=\Phi\left(\frac{\Phi^{-1}(u)-\rho\Phi^{-1}(v)}{\sqrt{1-\rho^2}}\right) 
$$

In [ ]:
def gaussian_conditional_cdf_torch(u, v, rho):
    eps = 1e-6

    u = torch.clamp(u, eps, 1 - eps)
    v = torch.clamp(v, eps, 1 - eps)

    z_u = normal_ppf(u)
    z_v = normal_ppf(v)

    return normal_cdf((z_u - rho * z_v) / torch.sqrt(1 - rho**2))

### **Linear interpolation**



$$
x_{\mathrm{left}} \le x \le
x_{\mathrm{right}}
$$



$$
y_{\mathrm{left}}
= y(x_{\mathrm{left}}),
\qquad
y_{\mathrm{right}} = y(x_{\mathrm{right}}).
$$

interpolation weight

$$
w
=
\frac{x - x_{\mathrm{left}}} {x_{\mathrm{right}} - x_{\mathrm{left}}}.
$$

interpolated value

$$
y(x)
\approx (1-w)y_{\mathrm{left}} + w y_{\mathrm{right}}.
$$

In [7]:
def interp_fixed_x_torch(x_values, x_grid, y_grid):
    idx = torch.searchsorted(x_grid, x_values) - 1 # x_values 가 들어갈 위치 찾아줌 (x_grid에서 가장 가까운 값 작은쪽으로 찾아줌)
    idx = torch.clamp(idx, 0, len(x_grid) - 2)

    x_left = x_grid[idx]
    x_right = x_grid[idx + 1]

    y_left = y_grid[idx]
    y_right = y_grid[idx + 1]

    weight = (x_values - x_left) / (x_right - x_left)

    return (1 - weight) * y_left + weight * y_right

In [12]:
x_gr = [0,1,2,3,4,5]
y_gr = [2,4,1,9,2,7]
x_values = torch.tensor([0.3, 1.5, 2.7], dtype=torch.float64)

In [15]:
x_grid = torch.tensor([0., 1., 2., 3.], dtype=torch.float64)
y_grid = torch.tensor([0., 10., 20., 30.], dtype=torch.float64)
x_values = torch.tensor([0.5, 1.5, 2.5], dtype=torch.float64)

interp_fixed_x_torch(x_values, x_grid, y_grid)

tensor([ 5., 15., 25.], dtype=torch.float64)

### **R-BP**
$$
p_i(x) =p_{i-1}(x) \left[ (1-\alpha_i) + \alpha_i c_\rho \left( P_{i-1}(x), P_{i-1}(x_i) \right) \right]
$$

$$
P_i(x) =(1-\alpha_i)P_{i-1}(x) + \alpha_i H_\rho \left( P_{i-1}(x), P_{i-1}(x_i) \right)
$$



In [6]:
def R_BP_density_torch(x, x_grid, rho, p0_grid, P0_grid):
    p = p0_grid.clone()
    P = P0_grid.clone()

    for i in range(len(x)):
        alpha = 1 / (i + 2)

        u = P # P_{i-1}(x_grid) : grid 전체에 대한 cdf값들 
        u_i = interp_fixed_x_torch(x[i:i+1], x_grid, P)[0] # P_{i-1}(x_i) : 관측값 x_i를 현재 cdf에 대입해서 uniform scale로 바꾼 값 

        c_rho = gaussian_copula_density_torch(u, u_i, rho)
        H_rho = gaussian_conditional_cdf_torch(u, u_i, rho)

        p = p * ((1 - alpha) + alpha * c_rho)
        P = (1 - alpha) * P + alpha * H_rho

    return p, P

## Estimate $\rho$ using ADAM

R-BP density with $\rho$
$$
p_n^{(\rho)}(x)
$$

Log likelihood 

$$
\ell(\rho)=\sum_{i=1}^{n}\log p_n^{(\rho)}(x_i)
$$



$$
\hat{\rho}=\arg\max_{\rho}\ell(\rho)
$$



$$
\hat{\rho}=\arg\min_{\rho}\left\{-\ell(\rho)\right\}
$$



In [19]:
def estimate_rho_adam(x, x_grid, p0_grid, P0_grid, lr=0.05, n_iter=300):
    x_torch = torch.tensor(x, dtype=torch.float64)
    x_grid_torch = torch.tensor(x_grid, dtype=torch.float64)
    p0_grid_torch = torch.tensor(p0_grid, dtype=torch.float64)
    P0_grid_torch = torch.tensor(P0_grid, dtype=torch.float64)

    theta = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)

    optimizer = torch.optim.Adam([theta], lr=lr)

    loss_list = []
    rho_list = []

    for _ in range(n_iter):
        optimizer.zero_grad() # 이전 계산 지우기

        eps = 1e-6
        rho = (1 - eps) * torch.sigmoid(theta)

        p_est, P_est = R_BP_density_torch(x=x_torch, x_grid=x_grid_torch, rho=rho, p0_grid=p0_grid_torch, P0_grid=P0_grid_torch)

        p_x = interp_fixed_x_torch(x_torch, x_grid_torch, p_est)
        p_x = torch.clamp(p_at_x, 1e-6, None)

        log_lik = torch.sum(torch.log(p_x))

        loss = -log_lik # loss 계산

        loss.backward() # loss 줄이는 theta 방향 계산하기
        optimizer.step() # theta를 한번 계산하기 

        loss_list.append(loss.item())  #tensor에서 숫자만 꺼내서 list에 추가
        rho_list.append(rho.item())

    rho_hat = rho_list[-1]

    return rho_hat, np.array(rho_list), np.array(loss_list)